In [4]:
#part a

In [5]:
bits = [1,0,1,0]

def decode(chromosome):
    x = 0
    for bit in chromosome:
        x = x * 2 + bit
    return x

decode(bits)
#converts the chromosome into an integer

10

In [6]:
def fitness(chromosome):
  x = decode(chromosome)
  final = -x*x + 14*x + 5
  return final

fitness(bits)
#returns f according to the given fitness function

45

In [33]:
c1 = [0,1,1,0]
c2 = [1,0,0,1]
c3 = [1,1,0,0]
c4 = [0,0,1,1]

d1 = decode(c1)
d2 = decode(c2)
d3 = decode(c3)
d4 = decode(c4)
print("decoded:", d1, d2, d3, d4)

f1 = fitness(c1)
f2 = fitness(c2)
f3 = fitness(c3)
f4 = fitness(c4)
print("fitness:", f1, f2, f3, f4)

decoded: 6 9 12 3
fitness: 53 50 29 38


In [8]:
import random

def roulette_select(population):

    total = 0
    for chr in population:
        total += fitness(chr) #compute total fitness

    r = random.random()
    cumulative = 0

    for chr in population:
        prob = fitness(chr) / total #add all probabilities we are getting
        cumulative += prob

        if r <= cumulative:
            return chr

#we add up the fitness of all chromosomes to get the total
#each chromosome gets a part of the wheel proportional to its fitness
#we pick a random number r between 0 and 1.
#we go through the chromosomes, adding their probabilities one by one
#the first chromosome whose probability is bigger than r gets selected at the end

pop = [
    [0,1,1,0],
    [1,0,0,1],
    [1,1,0,0],
    [0,0,1,1]
]

roulette_select(pop)
#selects a random parent according to the spin of the roulette wheel



[1, 1, 0, 0]

In [9]:
#crossover
#we take 2 parents and we cut them at a point and swap parts of each

def single_point_crossover(parent1, parent2, point):
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

In [10]:
p1 = pop[0]
p2 = pop[1]
c1, c2 = single_point_crossover(p1, p2, 2)

print(c1, c2)

[0, 1, 0, 1] [1, 0, 1, 0]


In [11]:
#mutation
#we will flip bits randomly
def mutate(chromosome, p_m):
    new_chrom = chromosome.copy()

    for i in range(len(new_chrom)):
        if random.random() < p_m:
            #flip bit
            new_chrom[i] = 1 - new_chrom[i]

    return new_chrom

In [12]:
for chrom in pop:
    mutated = mutate(chrom, 0.1)
    print("Original:", chrom, "Mutated:", mutated)

Original: [0, 1, 1, 0] Mutated: [0, 1, 1, 0]
Original: [1, 0, 0, 1] Mutated: [1, 0, 0, 0]
Original: [1, 1, 0, 0] Mutated: [1, 1, 0, 0]
Original: [0, 0, 1, 1] Mutated: [0, 0, 1, 1]


In [13]:
#part b

In [14]:
def run_ga(pop_size, num_generations, p_m, elitism=False):
    #initialize random population
    population = [[random.randint(0,1) for _ in range(4)] for _ in range(pop_size)]

    history = []

    for gen in range(num_generations):
        new_population = []

        if elitism:
            best_individual = max(population, key=fitness)

        #generate new population
        while len(new_population) < pop_size:

            p1 = roulette_select(population)
            p2 = roulette_select(population)

            #crossover point
            point = random.randint(1,3)
            c1, c2 = single_point_crossover(p1, p2, point)

            #mutation
            c1 = mutate(c1, p_m)
            c2 = mutate(c2, p_m)

            new_population.extend([c1, c2])

        population = new_population[:pop_size]

        #apply elitism: replace worst with best from previous gen
        if elitism:
            worst_index = min(range(pop_size), key=lambda i: fitness(population[i]))
            population[worst_index] = best_individual

        #record best individual of this generation
        best_ind = max(population, key=fitness)
        history.append((gen, fitness(best_ind), decode(best_ind)))

        print(f"\ngeneration {gen}:")
        print("population:", population)
        print("decoded x:", [decode(c) for c in population])
        print("fitness:", [fitness(c) for c in population])
        print("best individual:", best_ind, "x =", decode(best_ind))

    return history

In [15]:
history = run_ga(pop_size=4, num_generations=10, p_m=0.1, elitism=False)


generation 0:
population: [[0, 0, 1, 0], [0, 1, 1, 0], [0, 0, 1, 1], [1, 1, 1, 0]]
decoded x: [2, 6, 3, 14]
fitness: [29, 53, 38, 5]
best individual: [0, 1, 1, 0] x = 6

generation 1:
population: [[0, 0, 1, 1], [0, 1, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0]]
decoded x: [3, 6, 6, 6]
fitness: [38, 53, 53, 53]
best individual: [0, 1, 1, 0] x = 6

generation 2:
population: [[0, 0, 1, 1], [0, 1, 1, 0], [0, 1, 1, 1], [0, 1, 1, 0]]
decoded x: [3, 6, 7, 6]
fitness: [38, 53, 54, 53]
best individual: [0, 1, 1, 1] x = 7

generation 3:
population: [[0, 1, 1, 0], [1, 1, 1, 0], [0, 0, 1, 0], [0, 1, 1, 1]]
decoded x: [6, 14, 2, 7]
fitness: [53, 5, 29, 54]
best individual: [0, 1, 1, 1] x = 7

generation 4:
population: [[0, 1, 1, 0], [0, 0, 1, 1], [1, 1, 0, 1], [0, 1, 0, 0]]
decoded x: [6, 3, 13, 4]
fitness: [53, 38, 18, 45]
best individual: [0, 1, 1, 0] x = 6

generation 5:
population: [[0, 1, 1, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 1, 0]]
decoded x: [6, 4, 2, 6]
fitness: [53, 45, 29, 53]
best individual

In [16]:
#part c

In [31]:
import statistics
import random

best_fitness_list = []
found_optimum = 0
gen_to_50_list = [] #stores which generation reaches fitness ≥ 50 first in each trial

for i in range(30): #running 30 trials
    history = run_ga(pop_size=4, num_generations=20, p_m=0.1, elitism=False)

    #best fitness in this run
    best = max(f for (gen, f, x) in history)
    best_fitness_list.append(best)

    #check if x=7 found
    if any(x == 7 for (gen, f, x) in history):
        found_optimum += 1

    #first gen where f >= 50
    gens = [gen for (gen, f, x) in history if f >= 50]
    gen_to_50_list.append(min(gens) if gens else 20)

print("\nElitism = False")
print("Avg Best Fitness:", round(statistics.mean(best_fitness_list), 2))
print("Runs Found x=7:", found_optimum)
print("Avg Gen to f>=50:", round(statistics.mean(gen_to_50_list), 2))


generation 0:
population: [[1, 0, 1, 0], [1, 0, 1, 0], [0, 1, 0, 0], [1, 1, 1, 1]]
decoded x: [10, 10, 4, 15]
fitness: [45, 45, 45, -10]
best individual: [1, 0, 1, 0] x = 10

generation 1:
population: [[1, 0, 1, 0], [1, 0, 1, 0], [0, 0, 1, 0], [1, 0, 1, 1]]
decoded x: [10, 10, 2, 11]
fitness: [45, 45, 29, 38]
best individual: [1, 0, 1, 0] x = 10

generation 2:
population: [[1, 1, 1, 0], [1, 0, 1, 1], [1, 0, 1, 0], [1, 0, 0, 0]]
decoded x: [14, 11, 10, 8]
fitness: [5, 38, 45, 53]
best individual: [1, 0, 0, 0] x = 8

generation 3:
population: [[1, 0, 0, 0], [1, 0, 1, 1], [1, 0, 0, 1], [1, 0, 1, 0]]
decoded x: [8, 11, 9, 10]
fitness: [53, 38, 50, 45]
best individual: [1, 0, 0, 0] x = 8

generation 4:
population: [[1, 0, 1, 1], [0, 0, 0, 1], [1, 0, 1, 0], [1, 0, 0, 0]]
decoded x: [11, 1, 10, 8]
fitness: [38, 18, 45, 53]
best individual: [1, 0, 0, 0] x = 8

generation 5:
population: [[1, 0, 1, 1], [1, 0, 1, 0], [1, 0, 1, 1], [1, 0, 1, 0]]
decoded x: [11, 10, 11, 10]
fitness: [38, 45, 38, 4

In [24]:
best_fitness_list = []
found_optimum = 0
gen_to_50_list = []

for i in range(30): #only true different from above code
    history = run_ga(pop_size=4, num_generations=20, p_m=0.1, elitism=True)

    best = max(f for (gen, f, x) in history)
    best_fitness_list.append(best)

    if any(x == 7 for (gen, f, x) in history):
        found_optimum += 1

    gens = [gen for (gen, f, x) in history if f >= 50]
    gen_to_50_list.append(min(gens) if gens else 20)

print("\nElitism = True")
print("Avg Best Fitness:", round(statistics.mean(best_fitness_list), 2))
print("Runs Found x=7:", found_optimum)
print("Avg Gen to f>=50:", round(statistics.mean(gen_to_50_list), 2))


generation 0:
population: [[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 1, 0], [1, 0, 0, 1]]
decoded x: [5, 10, 6, 9]
fitness: [50, 45, 53, 50]
best individual: [0, 1, 1, 0] x = 6

generation 1:
population: [[0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 1, 0], [1, 0, 0, 1]]
decoded x: [4, 2, 6, 9]
fitness: [45, 29, 53, 50]
best individual: [0, 1, 1, 0] x = 6

generation 2:
population: [[0, 1, 1, 0], [1, 0, 0, 0], [0, 1, 0, 1], [0, 1, 1, 0]]
decoded x: [6, 8, 5, 6]
fitness: [53, 53, 50, 53]
best individual: [0, 1, 1, 0] x = 6

generation 3:
population: [[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 0], [1, 0, 0, 1]]
decoded x: [6, 10, 12, 9]
fitness: [53, 45, 29, 50]
best individual: [0, 1, 1, 0] x = 6

generation 4:
population: [[1, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [1, 0, 1, 1]]
decoded x: [8, 6, 6, 11]
fitness: [53, 53, 53, 38]
best individual: [1, 0, 0, 0] x = 8

generation 5:
population: [[1, 0, 0, 0], [0, 1, 1, 0], [1, 0, 0, 0], [1, 0, 1, 1]]
decoded x: [8, 6, 8, 11]
fitness: [53, 53, 53, 38]
best indivi

In [30]:
import statistics

mutation_rates = [0.01, 0.1, 0.3, 0.5]
num_trials = 30

results_pm = {}

for pm in mutation_rates:
    best_fitnesses = []

    for _ in range(num_trials):
        history = run_ga(pop_size=4, num_generations=20, p_m=pm, elitism=False)
        #get the best fitness in this run
        best_fitness = max(f for (_, f, _) in history)
        best_fitnesses.append(best_fitness)

    #save the average best fitness for this mutation rate
    results_pm[pm] = round(statistics.mean(best_fitnesses), 2)

    print("\nAverage Best Fitness per Mutation Rate (30 trials each)")
print("\nMutation Rate | Avg Best Fitness")
for pm, avg_fit in results_pm.items():
    print(f"{pm:<13} | {avg_fit}")


generation 0:
population: [[0, 1, 1, 1], [0, 1, 1, 1], [1, 0, 0, 1], [0, 1, 1, 1]]
decoded x: [7, 7, 9, 7]
fitness: [54, 54, 50, 54]
best individual: [0, 1, 1, 1] x = 7

generation 1:
population: [[0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1], [1, 0, 0, 1]]
decoded x: [7, 7, 7, 9]
fitness: [54, 54, 54, 50]
best individual: [0, 1, 1, 1] x = 7

generation 2:
population: [[0, 1, 1, 1], [0, 1, 1, 1], [1, 1, 1, 1], [0, 0, 0, 1]]
decoded x: [7, 7, 15, 1]
fitness: [54, 54, -10, 18]
best individual: [0, 1, 1, 1] x = 7

generation 3:
population: [[0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1]]
decoded x: [7, 7, 7, 7]
fitness: [54, 54, 54, 54]
best individual: [0, 1, 1, 1] x = 7

generation 4:
population: [[0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1]]
decoded x: [7, 7, 7, 7]
fitness: [54, 54, 54, 54]
best individual: [0, 1, 1, 1] x = 7

generation 5:
population: [[0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1]]
decoded x: [7, 7, 7, 7]
fitness: [54, 54, 54, 54]
best individua